In [ ]:
import hashlib
import json
import os
from collections import Counter, defaultdict

import cv2

base_path = r"C:\Users\Administrator\.cache\kagglehub\competitions\ai12-level1-project"
image_root = os.path.join(base_path, "sprint_ai_project1_data", "train_images")

image_files = [
    f for f in os.listdir(image_root) if f.endswith((".png", ".jpg", ".jpeg"))
]
print(f"전체 이미지 개수: {len(image_files)}")

# 1. MD5 중복 확인
md5_to_files = defaultdict(list)
for fname in image_files:
    fpath = os.path.join(image_root, fname)
    with open(fpath, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    md5_to_files[file_hash].append(fname)

duplicates = {h: files for h, files in md5_to_files.items() if len(files) > 1}
print("\n=== MD5 중복 이미지 ===")
print(f"중복 그룹 개수: {len(duplicates)}")
for h, files in list(duplicates.items())[:5]:
    print(f"  {files}")

# 2. Blur score 계산
blur_scores = []
for fname in image_files:
    fpath = os.path.join(image_root, fname)
    img = cv2.imread(fpath, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    score = cv2.Laplacian(img, cv2.CV_64F).var()
    blur_scores.append((fname, score))

blur_scores.sort(key=lambda x: x[1])
n = len(blur_scores)
bottom_5pct_count = max(1, int(n * 0.05))
bottom_5pct = blur_scores[:bottom_5pct_count]

print("\n=== Blur Score 분석 ===")
print(f"전체 이미지: {n}개")
print(f"하위 5% (블러 의심 후보): {bottom_5pct_count}개")
for fname, score in bottom_5pct:
    print(f"  {fname}: blur_score={score:.2f}")
print(
    f"\n전체 blur score 범위: 최소={blur_scores[0][1]:.2f}, 최대={blur_scores[-1][1]:.2f}"
)

# 3. 블러 후보 제거 시 클래스 영향 확인
merged_path = os.path.join(os.getcwd(), "merged_annotations_clean.json")
with open(merged_path, "r", encoding="utf-8") as fp:
    coco = json.load(fp)

blur_candidates = [fname for fname, score in bottom_5pct]
file_to_img_id = {img["file_name"]: img["id"] for img in coco["images"]}
category_id_to_name = {cat["id"]: cat["name"] for cat in coco["categories"]}
total_class_counts = Counter(ann["category_id"] for ann in coco["annotations"])

blur_img_ids = set(file_to_img_id[f] for f in blur_candidates if f in file_to_img_id)
affected_class_counts = Counter()
for ann in coco["annotations"]:
    if ann["image_id"] in blur_img_ids:
        affected_class_counts[ann["category_id"]] += 1

print("\n=== 블러 후보 제거 시 영향받는 클래스 ===")
for cat_id, count in affected_class_counts.items():
    name = category_id_to_name.get(cat_id, "??")
    total = total_class_counts[cat_id]
    remaining = total - count
    warning = " ⚠️ 위험 (희귀 클래스)" if remaining <= 3 else ""
    print(
        f"{name} (id={cat_id}): 전체 {total}개 중 {count}개 영향 → 제거 후 {remaining}개 남음{warning}"
    )
